# Nonidealities in Asynchronous Sigma-Delta Modulators

Simulation results for the nonidealities chapter.

The simulations of the asynchronous sigma-delta modulator chapter are repeated for a
quantizer with a delay. For an integrator, a first-order low-pass and a second-order
low-pass loop filter, the DC characteristic of the modulator is simulated and
characterized by the Taylor coefficients of the average output and of the oscillation
frequency. The simulated coefficients are compared with the predictions of the
nonlinear model of the nonidealities chapter.

In [ ]:
import os

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from delta_sigma_simulator.modulator import DeltaSigmaModulator
from delta_sigma_simulator.filter import (
    FilterIntegrator,
    FilterFirstOrder,
    FilterSecondOrder,
)
from delta_sigma_simulator.quantizer import QuantizerDelayHysteresis
from delta_sigma_simulator.wave import BinaryWave

In [ ]:
# - Directory the generated CSV files are written to. Set the BOOK_TIKZ environment
#   variable to the assets/tikz directory of the book to update its figures directly.
out = Path(os.environ.get("BOOK_TIKZ", "."))
# - Self-oscillation frequency
f0 = 1.0
# - Delay of the quantizer, a twentieth of the self-oscillation period. The delay is
#   bounded by f0 < 1 / (4 t_d) for the modulator to sustain this frequency.
t_d = 0.05 / f0
# - Ratio of the corner frequency of the loop filter to the self-oscillation frequency,
#   for the simulated and for the predicted values
beta0_sim = 1 / np.logspace(0, 3, 20)
beta0 = 1 / np.logspace(0, 3, 1000)
# - The general loop filter approximation requires a corner frequency well below the
#   self-oscillation frequency, so the second-order low-pass filter is only evaluated on
#   the lower half of that range
beta0_gen_sim = 1 / np.logspace(np.log10(2), 3, 20)
beta0_gen = 1 / np.logspace(np.log10(2), 3, 1000)
# - Amplitude of the average output used to extract the Taylor coefficients, and the
#   number of points it is sampled with. A larger amplitude improves the conditioning of
#   the fit, a smaller one reduces the influence of the higher-order terms.
V_max = 0.1
n_V = 21

## DC characteristic

For a static input, the modulator settles into a limit cycle. The oscillation
frequency `f` follows from the period of the limit cycle and the average output `V`
from its duty cycle. Sweeping the static input yields the DC characteristic of the
modulator, which is characterized by the Taylor coefficients `V_n` of the average
output with respect to the static input, and by the Taylor coefficients `f_n` of the
oscillation frequency with respect to the average output.

In [ ]:
def limit_cycle(asdm, U, n):
    """
    Simulate the modulator for a static input and return the oscillation frequency and
    the average output of the resulting limit cycle.

    Arguments:
        asdm: The modulator to simulate.
        U: The static input.
        n: The number of transitions to simulate.
    """
    e = asdm.simulate([BinaryWave(E=U)], n=n).e

    assert np.isclose(e[-1] - e[-3], e[-3] - e[-5]), "Simulation failed to converge."

    # - The output is +1 on the interval (e[-3], e[-2]) when it is preceded by an even
    #   number of transitions, and -1 otherwise
    s = (-1.0) ** (len(e) - 3)

    f = 1 / (e[-1] - e[-3])
    V = s * (2 * (e[-2] - e[-3]) / (e[-1] - e[-3]) - 1)

    return f, V


def dc_characteristic(asdm, U, n):
    """
    Simulate the DC characteristic of the modulator.

    Arguments:
        asdm: The modulator to simulate.
        U: Array of static inputs.
        n: The number of transitions to simulate per static input.
    """
    f = np.zeros_like(U)
    V = np.zeros_like(U)

    for i, U_i in enumerate(U):
        f[i], V[i] = limit_cycle(asdm, U_i, n)

    return f, V


def taylor(x, y, n=7):
    """
    Return the first n + 1 Taylor coefficients of y as a function of x, obtained by
    fitting a polynomial of degree n. The abscissa is normalized before fitting to keep
    the system well conditioned.

    Arguments:
        x: Array of abscissa values, centred around zero.
        y: Array of ordinate values.
        n: Degree of the polynomial.
    """
    x0 = np.max(np.abs(x))

    c = np.polynomial.polynomial.polyfit(x / x0, y, n)

    return c / x0 ** np.arange(n + 1)


def coefficients(U, V, f):
    """
    Taylor coefficients f_0, f_2, V_1 and V_3 of a simulated DC characteristic. The
    coefficients of the average output are obtained by inverting the series of the static
    input.

    Arguments:
        U: Array of static inputs.
        V: Array of average outputs.
        f: Array of oscillation frequencies.
    """
    c_f = taylor(V, f)
    c_U = taylor(V, U)

    return c_f[0], c_f[2], 1 / c_U[1], -c_U[3] / c_U[1] ** 4


def plot_amplitude(beta0, c, beta0_sim, c_sim):
    """
    Plot the predicted and the simulated value of V_1 and V_3.
    """
    plt.loglog(beta0, np.abs(c[3]), "k-", label=r"$V_1$")
    plt.loglog(beta0_sim, np.abs(c_sim[3]), "ko", label=r"$V_{1,\mathrm{sim}}$")
    plt.loglog(beta0, np.abs(c[4]), "-", color="gray", label=r"$V_3$")
    plt.loglog(
        beta0_sim, np.abs(c_sim[4]), "o", color="gray", label=r"$V_{3,\mathrm{sim}}$"
    )
    plt.xlabel(r"$\beta_0$")
    plt.ylabel(r"$\left|.\right|$")
    plt.grid(which="both", ls="--", alpha=0.5)
    plt.legend()
    plt.show()


def plot_frequency(beta0, c, beta0_sim, c_sim):
    """
    Plot the predicted and the simulated value of f_0 and f_2.
    """
    plt.semilogx(beta0, c[1], "k-", label=r"$f_0$")
    plt.semilogx(beta0_sim, c_sim[1], "ko", label=r"$f_{0,\mathrm{sim}}$")
    plt.semilogx(beta0, c[2] / c[1], "-", color="gray", label=r"$f_2/f_0$")
    plt.semilogx(
        beta0_sim,
        c_sim[2] / c_sim[1],
        "o",
        color="gray",
        label=r"$f_{2,\mathrm{sim}}/f_{0,\mathrm{sim}}$",
    )
    plt.xlabel(r"$\beta_0$")
    plt.ylim(-2, +2)
    plt.grid(which="both", ls="--", alpha=0.5)
    plt.legend()
    plt.show()

## General loop filter approximation

For a loop filter that is approximately first order at high frequencies, the nonlinear
model of this chapter gives the oscillation frequency in closed form as
`f = f0 (1 - V^2)`, so that `f_2 = -f0`, and the average output through the Taylor
coefficients

    V_1 = 1 / (1 + alpha0 / beta0 theta_g (1 + s theta_g / 2) - s x)
    V_3 = s x V_1^4,

where `x = pi^2 alpha0 beta0 / 6` and `theta_g = 2 pi beta0 f0 t_d` is the delay
normalized to the corner frequency of the loop filter. The sign `s` is that of the real
part of the loop filter transfer function at the self-oscillation frequency, which
selects the upper or the lower sign of the design equations, since the real part is
what the even part of the loop filter response follows. The terms that do not carry `s`
originate from the odd part of the response, which is unaffected by that sign.

In [ ]:
def general_coefficients(beta0, alpha0, s, theta_g):
    """
    Taylor coefficients V_1 and V_3 predicted by the general loop filter approximation.

    Arguments:
        beta0: Ratio of the corner frequency to the self-oscillation frequency.
        alpha0: Ratio of the gain at the self-oscillation frequency to the DC gain.
        s: Sign of the real part of the loop filter transfer function.
        theta_g: Delay normalized to the corner frequency of the loop filter.
    """
    x = np.pi**2 / 6 * alpha0 * beta0

    V_1 = 1 / (1 + alpha0 / beta0 * theta_g * (1 + s * theta_g / 2) - s * x)
    V_3 = s * x * V_1**4

    return V_1, V_3


def general_hysteresis(beta0, A_0, s, theta_g):
    """
    Hysteresis that yields the desired self-oscillation frequency.

    Arguments:
        beta0: Ratio of the corner frequency to the self-oscillation frequency.
        A_0: Gain of the loop filter at the self-oscillation frequency.
        s: Sign of the real part of the loop filter transfer function.
        theta_g: Delay normalized to the corner frequency of the loop filter.
    """
    return A_0 * (
        np.pi / 2 * (1 + s * theta_g) - theta_g / beta0 * (1 + s * theta_g / 2)
    )

## Integrator loop filter

The loop filter is an integrator with unity gain at the self-oscillation frequency.
Compared with the ideal modulator, the hysteresis has to be reduced by a factor
`1 - 4 f0 t_d` to sustain the same self-oscillation frequency. The predicted DC
characteristic is unchanged, so the modulator remains perfectly linear.

In [ ]:
# - Gain of the loop filter at the self-oscillation frequency
A_0 = 1.0
# - Hysteresis that yields the desired self-oscillation frequency
d = np.pi / 2 * A_0 * (1 - 4 * f0 * t_d)

c_int = np.stack(
    [
        1 / beta0,
        f0 * np.ones_like(beta0),
        -f0 * np.ones_like(beta0),
        np.ones_like(beta0),
        np.zeros_like(beta0),
    ]
)

c_int_sim = np.zeros((5, len(beta0_sim)))

for i, beta0_i in enumerate(beta0_sim):
    quantizer = QuantizerDelayHysteresis(t_d, d)
    quantizer.t_step = 0.1 / f0

    asdm = DeltaSigmaModulator(FilterIntegrator(2 * np.pi * f0 * A_0), quantizer)

    U = V_max * np.linspace(-1, 1, n_V)

    f, V = dc_characteristic(asdm, U, n=20)

    c_int_sim[:, i] = 1 / beta0_i, *coefficients(U, V, f)

np.savetxt(
    out / "non-ideal-loop-filter-integrator-expression.csv", c_int.T, delimiter=","
)
np.savetxt(
    out / "non-ideal-loop-filter-integrator-simulation.csv", c_int_sim.T, delimiter=","
)

In [ ]:
plot_amplitude(beta0, c_int, beta0_sim, c_int_sim)
plot_frequency(beta0, c_int, beta0_sim, c_int_sim)

## First-order low-pass loop filter

The loop filter is a first-order low-pass filter with unity DC gain and a corner
frequency `beta0 f0`. The delay reduces the gain of the modulator by a factor
`exp(-theta_g)`, while the oscillation frequency as a function of the average output is
unchanged. The prediction of the general loop filter approximation is evaluated as
well, for which the gain at the self-oscillation frequency is `beta0` times the DC gain
and the real part of the transfer function at the self-oscillation frequency is
positive.

In [ ]:
# - DC gain of the loop filter
A_DC = 1.0
# - Delay normalized to the corner frequency of the loop filter
theta_g = 2 * np.pi * beta0 * f0 * t_d

# - Exact prediction
w = np.pi * beta0 / 2 / np.tanh(np.pi * beta0 / 2)

V_1 = np.sinh(np.pi * beta0) / (np.pi * beta0) * np.exp(-theta_g)
V_3 = (np.pi**2 * beta0**2 / 12 + w * (w - 1)) * V_1**3

# - Prediction of the general loop filter approximation
V_1_gen, V_3_gen = general_coefficients(beta0, beta0, +1.0, theta_g)

c_lpf = np.stack(
    [1 / beta0, f0 * np.ones_like(beta0), -f0 * w, V_1, V_3, V_1_gen, V_3_gen]
)

c_lpf_sim = np.zeros((5, len(beta0_sim)))

for i, beta0_i in enumerate(beta0_sim):
    theta_g_i = 2 * np.pi * beta0_i * f0 * t_d

    # - Hysteresis that yields the desired self-oscillation frequency
    d = A_DC * (1 - np.exp(theta_g_i) * (1 - np.tanh(np.pi * beta0_i / 2)))

    quantizer = QuantizerDelayHysteresis(t_d, d)
    quantizer.t_step = 0.1 / f0

    asdm = DeltaSigmaModulator(FilterFirstOrder(beta0_i * f0, A_DC), quantizer)

    V_1_i = np.sinh(np.pi * beta0_i) / (np.pi * beta0_i) * np.exp(-theta_g_i)

    U = V_max / V_1_i * np.linspace(-1, 1, n_V)

    f, V = dc_characteristic(asdm, U, n=20)

    c_lpf_sim[:, i] = 1 / beta0_i, *coefficients(U, V, f)

np.savetxt(out / "non-ideal-loop-filter-low-pass-expression.csv", c_lpf.T, delimiter=",")
np.savetxt(
    out / "non-ideal-loop-filter-low-pass-simulation.csv", c_lpf_sim.T, delimiter=","
)

In [ ]:
plot_amplitude(beta0, c_lpf, beta0_sim, c_lpf_sim)
plot_frequency(beta0, c_lpf, beta0_sim, c_lpf_sim)

## Second-order low-pass loop filter

The loop filter is a second-order low-pass filter with unity DC gain, a zero at
`4 beta0 f0` and poles at `2 beta0 f0` and `beta0 f0`. Its gain at the self-oscillation
frequency is `beta0 / 2` times the DC gain, and the real part of its transfer function
at the self-oscillation frequency is negative. No exact prediction is available for
this loop filter, so the general loop filter approximation is used instead.

In [ ]:
# - DC gain of the loop filter
A_DC = 1.0
# - Sign of the real part of the transfer function at the self-oscillation frequency
s = -1.0
# - Delay normalized to the corner frequency of the loop filter
theta_g = 2 * np.pi * beta0_gen * f0 * t_d

V_1, V_3 = general_coefficients(beta0_gen, beta0_gen / 2, s, theta_g)

c_gen = np.stack(
    [
        1 / beta0_gen,
        f0 * np.ones_like(beta0_gen),
        -f0 * np.ones_like(beta0_gen),
        V_1,
        V_3,
    ]
)

c_gen_sim = np.zeros((5, len(beta0_gen_sim)))

for i, beta0_i in enumerate(beta0_gen_sim):
    theta_g_i = 2 * np.pi * beta0_i * f0 * t_d

    # - Hysteresis that yields the desired self-oscillation frequency
    d = general_hysteresis(beta0_i, beta0_i / 2 * A_DC, s, theta_g_i)

    quantizer = QuantizerDelayHysteresis(t_d, d)
    quantizer.t_step = 0.1 / f0

    asdm = DeltaSigmaModulator(
        FilterSecondOrder(4 * beta0_i * f0, 2 * beta0_i * f0, beta0_i * f0, A_DC),
        quantizer,
    )

    U = V_max * np.linspace(-1, 1, n_V)

    # - The limit cycle of a second-order loop filter settles over roughly half a time
    #   constant of its slowest pole, so the number of transitions is scaled accordingly
    f, V = dc_characteristic(asdm, U, n=int(max(20, 1 / beta0_i)))

    c_gen_sim[:, i] = 1 / beta0_i, *coefficients(U, V, f)

np.savetxt(out / "non-ideal-loop-filter-general-expression.csv", c_gen.T, delimiter=",")
np.savetxt(
    out / "non-ideal-loop-filter-general-simulation.csv", c_gen_sim.T, delimiter=","
)

In [ ]:
plot_amplitude(beta0_gen, c_gen, beta0_gen_sim, c_gen_sim)
plot_frequency(beta0_gen, c_gen, beta0_gen_sim, c_gen_sim)